In [1]:
import os
import sys
from pathlib import Path
import logging
import time

from websocket_server import WebsocketServer
from multiprocessing import Process

import pickle
import json
import pandas as pd
import numpy as np

cur_dir = os.getcwd()
path = Path(cur_dir)
sys.path.insert(0, str(path.parent.absolute()))

from src.preprocess import preprocess_df
from src.real_time_model import NetworkModel
from src.time_windowed import get_window
from src.safe_routing import communication_graph_from_df
from src.stream_data import start_web_socket_server, start_stream

In [2]:
with open(r'saves\victim_net.pickle', 'rb') as handle:
   names = pickle.load(handle) 

len(names)

13

In [3]:
with open(r'saves\connected_84.pickle', 'rb') as handle:
   names = pickle.load(handle) 

len(names)

84

In [8]:


df_raw = pd.read_csv('..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Tuesday-WorkingHours.pcap_ISCX.csv' , header=0, encoding='cp1252')
df = preprocess_df(df_raw, date_col=' Timestamp')

idx = df[' Source IP'].isin(names) & df[' Destination IP'].isin(names)
df = df[idx].copy()
df_temp = df.iloc[:10000,:]

In [4]:
df_temp

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
133451,192.168.10.16-192.168.10.50-137-137-17,192.168.10.50,137,192.168.10.16,137,17,2017-04-07 01:00:00,47,2,0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
133452,192.168.10.12-192.168.10.50-137-137-17,192.168.10.50,137,192.168.10.12,137,17,2017-04-07 01:00:00,3,2,0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
334175,192.168.10.50-192.168.10.51-21-53143-6,192.168.10.51,53143,192.168.10.50,21,6,2017-04-07 01:00:00,51,1,2,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
133516,192.168.10.16-192.168.10.50-58110-139-6,192.168.10.16,58110,192.168.10.50,139,6,2017-04-07 01:00:00,95,3,1,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
404986,192.168.10.3-192.168.10.5-445-56898-6,192.168.10.5,56898,192.168.10.3,445,6,2017-04-07 01:00:00,91534888,28,24,...,20,15644.0,31186.66682,62424.0,48.0,22900000.0,9215142.945,30000000.0,10700000.0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393898,192.168.10.3-192.168.10.51-53-9950-17,192.168.10.51,9950,192.168.10.3,53,17,2017-04-07 01:51:00,159,2,2,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
332328,192.168.10.3-192.168.10.17-53-10964-17,192.168.10.17,10964,192.168.10.3,53,17,2017-04-07 01:51:00,174,2,2,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
153197,172.217.10.142-192.168.10.16-443-44558-6,192.168.10.16,44558,172.217.10.142,443,6,2017-04-07 01:51:00,293622,12,11,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
429685,192.168.10.3-192.168.10.17-53-48661-17,192.168.10.17,48661,192.168.10.3,53,17,2017-04-07 01:51:00,23980,2,2,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [32]:
import src.network_connectivity, importlib

importlib.reload(src.network_connectivity)
from src.network_connectivity import*

In [33]:
import src.real_time_model, importlib

importlib.reload(src.real_time_model)
from src.real_time_model import*

In [34]:
import src.stream_data, importlib

importlib.reload(src.stream_data)
from src.stream_data import*

In [35]:
server = start_web_socket_server()

INFO:websocket_server.websocket_server:Listening on port 15674 for clients..
INFO:websocket_server.websocket_server:Starting WebsocketServer on thread Thread-12 (serve_forever).


Starting AMQP Server


In [37]:
start_stream(server, df, names, grow_entities=False, window_type='conn')

Waiting for Client


INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.


KeyboardInterrupt: 

INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.


In [38]:
server.clients



[]

In [39]:
server.disconnect_clients_gracefully()

In [40]:
server.shutdown_gracefully()

ERROR:websocket_server.websocket_server:********************************************************************************
Exception in child thread <WebsocketServerThread(Thread-12 (serve_forever), started daemon 35324)>: [WinError 10038] An operation was attempted on something that is not a socket
********************************************************************************
Traceback (most recent call last):
  File "C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\websocket_server\thread.py", line 27, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\bayer\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 232, in serve_forever
    ready = selector.select(poll_interval)
  File "C:\Users\bayer\AppData\Local\Programs\Python\Python310\lib\selectors.py", line 324, in select
    r, w, _ = self._select(self._readers, self._writers, [], timeout)
  File "C:\Users\bayer\AppData\Local\Programs\Python\Python310\lib\selectors.py", line 315, in _sele

In [41]:
server.shutdown_abruptly()

INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.
INFO:websocket_server.websocket_server:Client asked to close connection.


In [18]:
import src.network_connectivity, importlib

importlib.reload(src.network_connectivity)
from src.network_connectivity import*

In [21]:
del server